# Pediatric Chest X-ray Pneumonia Classification

Objective: train and compare a Custom CNN, ResNet50 transfer learning, and EfficientNetB0 transfer learning model for binary NORMAL vs PNEUMONIA classification using a clean patient-level train/validation/test split.

## Dataset Description

This project uses the pediatric chest X-ray dataset stored under `data/raw/chest_xray`. The official `test` directory is preserved unchanged. Training and validation CSVs are regenerated from the official train/validation pool using patient-level groups and SHA256 duplicate isolation.

## Setup and Dataset Loading

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SPLITS_DIR = PROJECT_ROOT / 'data' / 'splits'
REPORTS_DIR = PROJECT_ROOT / 'data' / 'reports'

train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'val.csv')
test_df = pd.read_csv(SPLITS_DIR / 'test.csv')
train_df.head()

## Clean Patient-Level Train/Validation/Test Split

The clean split is loaded from `data/splits/train.csv`, `data/splits/val.csv`, and `data/splits/test.csv`. PNEUMONIA images are grouped by `person<ID>`. NORMAL images are grouped by stable `IM-XXXX` or `NORMAL2-IM-XXXX` filename prefixes. SHA256 hashes are included for every image.

In [ ]:
for name, df in {'train': train_df, 'val': val_df, 'test': test_df}.items():
    print(name, 'images=', len(df), 'groups=', df['patient_group'].nunique(), 'hashes=', df['sha256'].nunique())

## Leakage Verification

In [ ]:
def overlap(a, b, col):
    return len(set(a[col]) & set(b[col]))

verification = {
    'train_val_group_overlap': overlap(train_df, val_df, 'patient_group'),
    'train_val_hash_overlap': overlap(train_df, val_df, 'sha256'),
    'train_test_group_overlap': overlap(train_df, test_df, 'patient_group'),
    'val_test_group_overlap': overlap(val_df, test_df, 'patient_group'),
    'train_test_hash_overlap': overlap(train_df, test_df, 'sha256'),
    'val_test_hash_overlap': overlap(val_df, test_df, 'sha256'),
}
verification

## Class Distribution

In [ ]:
dist = pd.concat([df.assign(prepared_split=name) for name, df in {'train': train_df, 'val': val_df, 'test': test_df}.items()])
class_counts = pd.crosstab(dist['prepared_split'], dist['label'])
display(class_counts)
class_counts.plot(kind='bar', figsize=(7, 4), title='Class distribution by split')
plt.ylabel('Images')
plt.tight_layout()

## Data Visualization

In [ ]:
from PIL import Image

def show_samples(df, title, n=4):
    sample = df.groupby('label', group_keys=False).apply(lambda x: x.sample(min(n, len(x)), random_state=42))
    fig, axes = plt.subplots(2, n, figsize=(12, 6))
    axes = np.array(axes).reshape(2, n)
    for ax in axes.ravel():
        ax.axis('off')
    for row_idx, label in enumerate(['NORMAL', 'PNEUMONIA']):
        label_rows = sample[sample['label'] == label].head(n).reset_index(drop=True)
        for col_idx, (_, row) in enumerate(label_rows.iterrows()):
            img = Image.open(PROJECT_ROOT / row['filepath']).convert('L')
            axes[row_idx, col_idx].imshow(img, cmap='gray')
            axes[row_idx, col_idx].set_title(label)
    fig.suptitle(title)
    plt.tight_layout()

show_samples(train_df, 'Training samples')

## Preprocessing

Images are decoded as RGB, resized to 224 x 224, and scaled to `[0, 1]`. The project implementation is reused from `src.preprocessing` instead of duplicating large data-pipeline code.

In [ ]:
import sys
sys.path.append(str(PROJECT_ROOT))
from src.preprocessing import load_prepared_datasets, compute_class_weights, make_augmentation_model

train_ds, val_ds, test_ds, class_weights, dfs = load_prepared_datasets(batch_size=32)
class_weights

## Data Augmentation

Training uses conservative medical-image augmentation: horizontal flip, small rotation, zoom, translation, and contrast adjustment. Validation and test data are not augmented.

In [ ]:
augmentation = make_augmentation_model()
augmentation.summary()

## Class Weights

In [ ]:
class_weights = compute_class_weights(dfs['train'])
class_weights

## Custom CNN Training Section

In [ ]:
# Full training script, using the clean split CSVs:
# !python -m src.train_custom_cnn --epochs 15 --batch-size 32

from src.models import build_custom_cnn
custom_cnn = build_custom_cnn()
custom_cnn.summary()

## ResNet50 Transfer Learning Section

In [ ]:
# Full training script, using the clean split CSVs:
# !python -m src.train_resnet50 --head-epochs 4 --fine-tune-epochs 3 --batch-size 32

from src.models import build_resnet50_transfer
resnet50_model = build_resnet50_transfer()
resnet50_model.summary()

## EfficientNetB0 Transfer Learning Section

In [ ]:
# Full training script, using the clean split CSVs:
# !python -m src.train_efficientnetb0 --head-epochs 4 --fine-tune-epochs 3 --batch-size 32

from src.models import build_efficientnetb0_transfer
efficientnet_model = build_efficientnetb0_transfer()
efficientnet_model.summary()

## Evaluation Metrics

Required metrics: Accuracy, Precision, Recall, F1-score, and AUC-ROC. After retraining on the clean split, load each model's prediction CSV or run inference on `test_ds`.

In [ ]:
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'auc_roc': roc_auc_score(y_true, y_prob),
    }

## Confusion Matrices

In [ ]:
def plot_confusion(y_true, y_prob, title, threshold=0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['NORMAL', 'PNEUMONIA']).plot(cmap='Blues')
    plt.title(title)
    plt.tight_layout()
    return cm

## ROC Curves

In [ ]:
def plot_roc(y_true, y_prob, label):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_value = roc_auc_score(y_true, y_prob)
    plt.plot(fpr, tpr, label=f'{label} AUC={auc_value:.3f}')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()

## Final Model Comparison

After clean-split retraining, compare Custom CNN, ResNet50, and EfficientNetB0 using the same official test set and the metrics above. Previous model-selection results are outdated because the validation split changed.

In [ ]:
report_files = {
    'Custom CNN': REPORTS_DIR / 'custom_cnn_evaluation.json',
    'ResNet50': REPORTS_DIR / 'resnet50_evaluation.json',
    'EfficientNetB0': REPORTS_DIR / 'efficientnetb0_evaluation.json',
}
rows = []
for model_name, path in report_files.items():
    if path.exists():
        report = json.loads(path.read_text())
        rows.append({'model': model_name, **report.get('test_metrics', {})})
comparison_df = pd.DataFrame(rows)
comparison_df

## Grad-CAM Explainability

Grad-CAM is used as an interpretability aid to inspect image regions influencing model predictions. It is not medical proof and should not be used as a diagnostic explanation by itself.

In [ ]:
# Existing Grad-CAM utility can be reused after a clean-split model is trained:
# !python -m src.gradcam

from pathlib import Path
gradcam_dir = PROJECT_ROOT / 'results' / 'gradcam'
list(gradcam_dir.glob('**/*overlay.png'))[:5] if gradcam_dir.exists() else []

## Limitations

- Academic prototype only.
- The dataset is pediatric and does not represent all patient populations.
- This is not a clinical diagnostic tool.
- Grad-CAM is an interpretability aid, not medical proof.
- Model-selection results produced before the clean patient-level split are outdated and must not be reported as final.

## Final Conclusion

The project now uses a patient-level train/validation split with exact image hash leakage checks and the official test set preserved. Custom CNN, ResNet50, and EfficientNetB0 must be retrained with the new clean split before final academic reporting.

<!-- FINAL_CLEAN_SPLIT_UPDATE -->
# Final Clean-Split Update

The final training run uses the leakage-fixed split from `data/splits/`. Patient/group and hash overlap were removed before model selection, while the official test set was preserved.

## Clean-Split Metrics

| Rank | Model | Accuracy | Precision | Recall | F1-score | AUC-ROC | Best F1 threshold | Confusion matrix |
|---:|---|---:|---:|---:|---:|---:|---:|---|
| 1 | EfficientNetB0 | `0.887821` | `0.875587` | `0.956410` | `0.914216` | `0.957868` | `0.60` | `TN=181, FP=53, FN=17, TP=373` |
| 2 | ResNet50 | `0.876603` | `0.861432` | `0.956410` | `0.906440` | `0.954175` | `0.85` | `TN=174, FP=60, FN=17, TP=373` |
| 3 | Custom CNN | `0.826923` | `0.813333` | `0.938462` | `0.871429` | `0.907287` | `0.70` | `TN=150, FP=84, FN=24, TP=366` |

## Final Winner

The final selected model is **EfficientNetB0** with accuracy `0.887821`, recall `0.956410`, F1-score `0.914216`, and AUC-ROC `0.957868`.

## Leakage Discussion

Previous model-selection results are treated as outdated because patient-level or duplicate leakage can inflate validation evidence. Final conclusions use only the clean-split retraining artifacts.

## Pediatric Dataset Limitation

The dataset is pediatric chest X-ray data. Results should not be generalized to adult populations, other hospitals, different machines, or other acquisition protocols without external validation.

## Figure References

- Training histories: `results/figures/custom_cnn_training_curves.png`, `results/figures/resnet50_training_curves.png`, `results/figures/efficientnetb0_training_curves.png`
- Confusion matrices: `results/confusion_matrices/`
- ROC curves: `results/roc_curves/`
- Grad-CAM summary: `results/figures/gradcam_summary_grid.png`
